In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_json("../../../data/post_quality/raw/post_data_v4.json")
df.reset_index(drop=True)
df.head()

,id,title,body,effort,openness,is_confident,subreddit,tag
0,1,If your first-ever attempt at gambling went co...,,0,0,True,r/Showerthoughts,NaN
1,2,"According to birds, all land animals are botto...",,0,0,True,r/Showerthoughts,NaN
2,3,Very few people can actually keep their watch ...,,0,0,True,r/Showerthoughts,NaN
3,4,"What is keeping the really deadly diseases, li...",,0,0,True,r/askscience,NaN
4,5,When did humans start living indoors?,,0,0,True,r/askscience,NaN


In [3]:
df.shape

(240, 8)

# Checking for null values and dropping the unrelevent column

In [4]:
for col in df.columns:
    print(f"Number of null value in '{col}' column is {df[col].isnull().sum()}")

Number of null value in 'id' column is 0
Number of null value in 'title' column is 0
Number of null value in 'body' column is 0
Number of null value in 'effort' column is 0
Number of null value in 'openness' column is 0
Number of null value in 'is_confident' column is 0
Number of null value in 'subreddit' column is 0
Number of null value in 'tag' column is 239


In [5]:
df.drop(columns=['tag','openness',"id"],inplace=True)

In [6]:
df.shape

(240, 5)

In [7]:
for col in df.columns:
    print(f"Number of null value in '{col}' column is {df[col].isnull().sum()}")

Number of null value in 'title' column is 0
Number of null value in 'body' column is 0
Number of null value in 'effort' column is 0
Number of null value in 'is_confident' column is 0
Number of null value in 'subreddit' column is 0


# Try Getting baseline feature set like this to just check whether any signals are there or not. I will include features are:
1. Structural
    - num_tokens: No. of words in each post
    - num_paragraphs: No. of paragraphs
    - has_multiple_paragraphs: A boolean if it is a multiple para post
    - avg_sentence_length: average length of sentence in each post
2. Intent/framing:
    - has_first_person: How much personal context, and ownership is there in the problem
    - has_attempt_marker: It tells about the reflection of past attempts
    - has_constraint_marker: It tell is there any constraints or trade-off
    - question_count: How many questions are there

In [8]:
import re
def count_paragraphs(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0
    return len([p for p in text.split('\n') if p.strip()])

df['num_paragraphs'] = df['body'].apply(count_paragraphs)
df['has_multi_paragraphs'] = df['num_paragraphs'] >= 2


df['text'] = df['title'].fillna('') + ' ' + df['body'].fillna('')

## Finding average sentence length

def avg_sentence_length(text):
    # split into sentences
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    if not sentences:
        return 0
    
    # count words per sentence
    word_counts = [len(s.split()) for s in sentences]
    return sum(word_counts) / len(word_counts)

df['avg_sentence_length'] = df['text'].apply(avg_sentence_length)

def count_tokens(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

df['num_tokens'] = df['text'].apply(count_tokens)




In [9]:
df.tail()

,title,body,effort,is_confident,subreddit,num_paragraphs,has_multi_paragraphs,text,avg_sentence_length,num_tokens
235,Does being an introvert actually lead to highe...,There’s a common belief that introverts tend t...,1,True,r/TrueAskReddit,4,True,Does being an introvert actually lead to highe...,15.833333,95
236,Does anything seem legendary anymore?,I was having a conversation with a friend as t...,1,True,r/TrueAskReddit,5,True,Does anything seem legendary anymore? I was ha...,14.062500,225
237,How will US government deal with a large propo...,How will US government deal with a large propo...,1,True,r/TrueAskReddit,3,True,How will US government deal with a large propo...,14.333333,129
238,Why so many straight men are constantly polici...,I’ve noticed a pattern where straight guys lov...,1,True,r/TrueAskReddit,2,True,Why so many straight men are constantly polici...,13.857143,97
239,Are we done?,Imagine the year is 2050 AI has evolved into A...,1,True,r/TrueAskReddit,3,True,Are we done? Imagine the year is 2050 AI has e...,20.800000,104


In [10]:
## First person feature
FIRST_PERSON_PATTERN = re.compile(r'\b(i|my|me|mine|i\'ve|i\'m|i am)\b',re.IGNORECASE)

df['has_first_person'] = df['text'].apply(
    lambda x: bool(FIRST_PERSON_PATTERN.search(x)) if isinstance(x,str) else False
)

## Attempts markers
ATTEMPT_MARKERS = ["tried","attempted","failed","didn't work","doesn't work","already tried", "so far"]

def has_attempt_marker(text):
    if not isinstance(text, str):
        return False
    text = text.lower()
    return any(marker in text for marker in ATTEMPT_MARKERS)

df['has_attempted_marker'] = df['text'].apply(has_attempt_marker)


## Checking for constraints
CONSTRAINT_MARKERS = [
    "but", "however", "because", "although", "though",
    "except", "unless", "while"
]

def has_constraint_marker(text):
    if not isinstance(text, str):
        return False
    text = text.lower()
    return any(f" {marker} " in text for marker in CONSTRAINT_MARKERS)

df['has_constraint_marker'] = df['text'].apply(has_constraint_marker)


## Counting questions
df['question_count'] = df['text'].str.count(r'\?')


In [11]:
df.head()

,title,body,effort,is_confident,subreddit,num_paragraphs,has_multi_paragraphs,text,avg_sentence_length,num_tokens,has_first_person,has_attempted_marker,has_constraint_marker,question_count
0,If your first-ever attempt at gambling went co...,,0,True,r/Showerthoughts,0,False,If your first-ever attempt at gambling went co...,8.0,15,False,False,False,0
1,"According to birds, all land animals are botto...",,0,True,r/Showerthoughts,0,False,"According to birds, all land animals are botto...",9.0,9,False,False,False,0
2,Very few people can actually keep their watch ...,,0,True,r/Showerthoughts,0,False,Very few people can actually keep their watch ...,13.0,13,False,False,False,0
3,"What is keeping the really deadly diseases, li...",,0,True,r/askscience,0,False,"What is keeping the really deadly diseases, li...",15.0,15,False,False,False,1
4,When did humans start living indoors?,,0,True,r/askscience,0,False,When did humans start living indoors?,6.0,6,False,False,False,1


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  240 non-null    object 
 1   body                   240 non-null    object 
 2   effort                 240 non-null    int64  
 3   is_confident           240 non-null    bool   
 4   subreddit              240 non-null    object 
 5   num_paragraphs         240 non-null    int64  
 6   has_multi_paragraphs   240 non-null    bool   
 7   text                   240 non-null    object 
 8   avg_sentence_length    240 non-null    float64
 9   num_tokens             240 non-null    int64  
 10  has_first_person       240 non-null    bool   
 11  has_attempted_marker   240 non-null    bool   
 12  has_constraint_marker  240 non-null    bool   
 13  question_count         240 non-null    int64  
dtypes: bool(5), float64(1), int64(4), object(4)
memory usage: 

In [13]:
df.drop(columns=['is_confident'],axis=1,inplace=True)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  240 non-null    object 
 1   body                   240 non-null    object 
 2   effort                 240 non-null    int64  
 3   subreddit              240 non-null    object 
 4   num_paragraphs         240 non-null    int64  
 5   has_multi_paragraphs   240 non-null    bool   
 6   text                   240 non-null    object 
 7   avg_sentence_length    240 non-null    float64
 8   num_tokens             240 non-null    int64  
 9   has_first_person       240 non-null    bool   
 10  has_attempted_marker   240 non-null    bool   
 11  has_constraint_marker  240 non-null    bool   
 12  question_count         240 non-null    int64  
dtypes: bool(4), float64(1), int64(4), object(4)
memory usage: 17.9+ KB


## Freezing the feature list and let's test it with logistic regression and CV

### Bunch of helper function

In [15]:
## Let's build a helper function to make error_df
import pandas as pd

def build_error_df(df_test, y_true, y_pred, model_name):
    out = df_test.copy()
    out['true_label'] = y_true
    out['pred_label'] = y_pred
    out['correct'] = y_true == y_pred
    out['model'] = model_name
    return out

In [16]:
## Generic function to split df in same test and train
from sklearn.model_selection import train_test_split


def split_df(
    df,
    label_col,
    test_size=0.2,
    random_state=52,
    stratify=True
):
    """
    Splits dataframe into train and test sets.

    Returns
    -------
    df_train : pd.DataFrame
    df_test : pd.DataFrame
    """

    df = df.reset_index(drop=True)

    stratify_col = df[label_col] if stratify else None

    train_idx, test_idx = train_test_split(
        df.index,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_col
    )

    df_train = df.loc[train_idx]
    df_test = df.loc[test_idx]

    return df_train, df_test


In [17]:
## Running numerical only model
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score


def run_numerical_model(
    df_train,
    df_test,
    label_col,
    num_cols=None,
    model=None,
    scaler=None,
    model_name="numerical_only"
):
    """
    Trains and evaluates a numerical-only classification model.

    Returns
    -------
    dict with:
        - model
        - scaler
        - classification_report
        - errors_df
        - y_pred
    """

    if num_cols is None:
        num_cols = df_train.select_dtypes(include="number").columns.tolist()
        num_cols.remove(label_col)

    if scaler is None:
        scaler = StandardScaler()

    if model is None:
        model = LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )

    X_train = scaler.fit_transform(df_train[num_cols])
    X_test = scaler.transform(df_test[num_cols])

    y_train = df_train[label_col]
    y_test = df_test[label_col]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred)

    errors_df = build_error_df(
        df_test=df_test,
        y_true=y_test,
        y_pred=y_pred,
        model_name=model_name
    )

    errors_df['error_type'] = ''
    errors_df['error_subtype'] = ''
    errors_df['notes'] = ''

    return {
        "model": model,
        "scaler": scaler,
        "classification_report": report,
        "errors_df": errors_df,
        "accuracy":accuracy_score(y_test,y_pred),
        "y_pred": y_pred
    }


In [18]:
## Function to get error across k-folds
import pandas as pd
from sklearn.model_selection import StratifiedKFold

def run_k_fold_error_analysis(df:pd.DataFrame, n_splits=5, label_col='label'):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )
    
    all_errors = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df[label_col])):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        
        result = run_numerical_model(
            train_df,
            test_df,
            label_col=label_col
        )
        
        errors_df = result['errors_df'].copy()
        
        errors_df['fold'] = fold
        
        all_errors.append(errors_df)
    
    errors_all = pd.concat(all_errors).reset_index(drop=True)
    
    errors_only = errors_all[errors_all['correct'] == False]
    
    errors_all['error_type'] = 'N/A'  
    errors_all['error_subtype'] = 'N/A'  
    errors_all['notes'] = ''  
    
    true_label_counts = errors_only['true_label'].value_counts()
    pred_label_counts = errors_only['pred_label'].value_counts()
    error_group_counts = errors_only.groupby(['true_label', 'pred_label']).size()
    
    return {
        'errors_all': errors_all,
        'errors_only': errors_only,
        'true_label_counts': true_label_counts,
        'pred_label_counts': pred_label_counts,
        'error_group_counts': error_group_counts
    }


In [19]:
result_v1 = run_k_fold_error_analysis(df=df,label_col='effort')

In [20]:
result_v1['errors_only']

,title,body,effort,subreddit,num_paragraphs,has_multi_paragraphs,text,avg_sentence_length,num_tokens,has_first_person,...,has_constraint_marker,question_count,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
9,Feeling frustrated over school taxes right now...,Yeah I know this is gonna sound like whining.....,0,r/frustration,4,True,Feeling frustrated over school taxes right now...,93.750000,375,True,...,True,1,0,1,False,numerical_only,,,,0
12,Java or Javascript or C#?,"I'm not sure which one if these is better, to ...",1,r/learningprogramming,1,False,Java or Javascript or C#? I'm not sure which o...,13.600000,67,True,...,True,2,1,0,False,numerical_only,,,,0
13,Urgent question on London flight connection,I have found a aer lingus flight from Dublin t...,1,r/travel,1,False,Urgent question on London flight connection I ...,12.833333,75,True,...,False,0,1,0,False,numerical_only,,,,0
18,Want to start learning python,I just thought of finally getting into this af...,1,r/learnpython,1,False,Want to start learning python I just thought o...,33.000000,99,True,...,True,2,1,0,False,numerical_only,,,,0
21,Trying to understand why large transmission ge...,On a car a smaller gear has more torque and a ...,1,r/AskEngineers,4,True,Trying to understand why large transmission ge...,12.142857,85,True,...,True,1,1,0,False,numerical_only,,,,0
23,When and why did athletes in team sports start...,As the question states; curious about why this...,0,r/AskHistorians,1,False,When and why did athletes in team sports start...,11.200000,56,False,...,False,4,0,1,False,numerical_only,,,,0
38,CMV: All ICE agents should go to prison,What I'm talking about isn't the question of a...,1,r/changemyview,5,True,CMV: All ICE agents should go to prison What I...,23.200000,232,True,...,True,0,1,0,False,numerical_only,,,,0
40,Which ethical or moral rules people treat as o...,I’ve been thinking about “obvious” moral rules...,1,r/TrueAskReddit,2,True,Which ethical or moral rules people treat as o...,13.714286,96,True,...,True,2,1,0,False,numerical_only,,,,0
56,i physically cannot see other humans as the sa...,"TW for rape, csa, incest and cult mentions but...",0,r/offmychest,7,True,i physically cannot see other humans as the sa...,34.062500,545,True,...,True,0,0,1,False,numerical_only,,,,1
57,Yesterday I was the medical emergency on a flight,Yesterday I got on a flight from London to Tor...,0,r/travel,14,True,Yesterday I was the medical emergency on a fli...,14.468750,463,True,...,True,0,0,1,False,numerical_only,,,,1


## Removing duplicates along the errors



In [21]:
errors = result_v1['errors_only']

# assuming you have some unique id or text
errors_unique = errors.drop_duplicates(subset=['title', 'body'])

len(errors), len(errors_unique)


(59, 59)

### It means there is no repeetition of a type of error in all 5-folds.
### So let's focus now in manually reading around 25 errors and try figuring out somethings

In [22]:
errors = result_v1['errors_only']
sample = errors.sample(25, random_state=42)


In [23]:
sample

,title,body,effort,subreddit,num_paragraphs,has_multi_paragraphs,text,avg_sentence_length,num_tokens,has_first_person,...,has_constraint_marker,question_count,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
9,Feeling frustrated over school taxes right now...,Yeah I know this is gonna sound like whining.....,0,r/frustration,4,True,Feeling frustrated over school taxes right now...,93.750000,375,True,...,True,1,0,1,False,numerical_only,,,,0
23,When and why did athletes in team sports start...,As the question states; curious about why this...,0,r/AskHistorians,1,False,When and why did athletes in team sports start...,11.200000,56,False,...,False,4,0,1,False,numerical_only,,,,0
156,Which companies now only offer on site softwar...,Hi all- currently work remotely and I now real...,1,r/cscareerquestions,1,False,Which companies now only offer on site softwar...,14.500000,58,True,...,False,2,1,0,False,numerical_only,,,,3
67,[IWantOut] 28M Chef USA -> Italy,I'm a chef in New York with 14 years experienc...,1,r/iWantOut,1,False,[IWantOut] 28M Chef USA -> Italy I'm a chef in...,19.500000,117,True,...,False,0,1,0,False,numerical_only,,,,1
182,Do antinatalists have a positive duty to abort...,By abort I don't mean aborting a pregnancy spe...,1,r/askphilosophy,1,False,Do antinatalists have a positive duty to abort...,19.250000,77,True,...,True,2,1,0,False,numerical_only,,,,3
213,ELI5 What happens to the heat when you put lot...,There is an Instagram trend in northern countr...,1,r/explainlikeimfive,1,False,ELI5 What happens to the heat when you put lot...,10.166667,61,False,...,False,2,1,0,False,numerical_only,,,,4
231,Is it unethical to steal from large corporatio...,I've been having this debate with a few friend...,1,r/Ethics,3,True,Is it unethical to steal from large corporatio...,21.800000,109,True,...,True,1,1,0,False,numerical_only,,,,4
113,My neighbors row home is dusty and it’s my fault?,I live in a row home. Well actually my parents...,1,r/HomeImprovement,2,True,My neighbors row home is dusty and it’s my fau...,15.636364,172,True,...,True,2,1,0,False,numerical_only,,,,2
198,TIFU by assuming I was safe to do a wee (pee) ...,One of my favourite things to do for fun an ex...,0,r/tifu,4,True,TIFU by assuming I was safe to do a wee (pee) ...,26.888889,482,True,...,True,2,0,1,False,numerical_only,,,,4
65,Why is there mold on my ceiling?,"Hi, having lived in the same house for 15 year...",1,r/HomeImprovement,1,False,"Why is there mold on my ceiling? Hi, having li...",13.833333,83,True,...,False,2,1,0,False,numerical_only,,,,1


## 🔍 Error Analysis & Model Learning (Effort Axis)

### Dataset & Baseline

- Trained a baseline numerical model:
  - Logistic regression
  - Class balancing enabled
- Dataset:
  - 240 manually labeled discussion posts
- Features:
  - Structurally interpretable features derived from post **title** and **body**
- Goal:
  - Not peak accuracy
  - Understand failure modes
  - Improve **recall on low-effort content**
  - Maintain **precision on high-effort posts**

**Result:**  
- Moderate accuracy (~70%)
- Consistent, interpretable errors surfaced

---

### Error Analysis Methodology

Rather than tuning hyperparameters or adding embeddings prematurely, I performed **systematic k-fold error analysis**:

- 5-fold stratified cross-validation
- Collected all false predictions across folds
- Manually inspected **25 representative misclassified posts**
- Grouped errors by **reason for failure**, not surface attributes

**Rationale:**  
This approach enabled reasoning about *why* the model failed, not just *how often*.

---

### Key Error Buckets Identified

Misclassifications clustered into three dominant categories:

| Bucket | Description                                                         | Share of Errors |
|------|----------------------------------------------------------------------|-----------------|
| A    | Concise but well-contextualized posts misclassified as low effort   | ~68%            |
| B    | Label ambiguity (borderline effort by guideline)                    | ~16%            |
| C    | Surface structure bias (length/questions mistaken for effort)       | ~12%            |

---

### Insight: What the Model Was Actually Learning

The baseline model relied heavily on **surface verbosity signals**, including:

- Paragraph count
- Token count
- Sentence length
- Question frequency

This caused systematic failure on posts that were:

- Short  
- But clearly contextualized  
- And framed a concrete situation, constraint, or motivation  

**Observation:**  
- Humans labeled these posts as `Effort = 1`
- The model lacked a signal for **contextual grounding**

---

### Design Decision

Instead of adding more generic features or switching to deep models, I chose to:

- Target **Bucket A** specifically
- Add a **minimal, interpretable signal** aligned with labeling guidelines
- Measure which predictions actually changed due to that signal

**Outcome:**  
This keeps the system:

- Debuggable
- Resume-explainable
- Extensible to other moderation tasks


# Helper function to compare result without feature and with feature.

In [24]:
def evaluate_feature_impact(
    df,
    label_col,
    feature_cols,
    n_splits=5,
    random_state=42
):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    changed_rows = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df[label_col])):
        train_df = df.iloc[train_idx]
        test_df = df.iloc[test_idx]

        # WITH feature
        result_with = run_numerical_model(
            train_df,
            test_df,
            label_col=label_col,
            num_cols=feature_cols
        )

        # WITHOUT feature
        reduced_cols = [c for c in feature_cols if c != feature_cols[-1]]
        result_without = run_numerical_model(
            train_df,
            test_df,
            label_col=label_col,
            num_cols=reduced_cols
        )

        pred_with = result_with['y_pred']
        pred_without = result_without['y_pred']

        diff_mask = pred_with != pred_without

        if diff_mask.any():
            diff_df = test_df.loc[diff_mask, ['title', 'body']].copy()
            diff_df['true_label'] = test_df.loc[diff_mask, label_col]
            diff_df['pred_without'] = pred_without[diff_mask]
            diff_df['pred_with'] = pred_with[diff_mask]
            diff_df['fold'] = fold
            diff_df['targetted_label'] = feature_cols[-1]

            changed_rows.append(diff_df)

    return pd.concat(changed_rows) if changed_rows else pd.DataFrame()


## Add a feature : `has_contextual_grounding`
**Meaningful effort includes context, constraints, reflection, or intent.**

It helps me to fix the largest error bucket

In [25]:
CONTEXT_MARKERS = ["because","due to","as a result","currently", "right now", "i am","i'm", "i work", "i live", "based in", "looking for", "trying to", "in my role", "in my situation"]

def has_contextual_grounding(text):
    if not isinstance(text, str):
        return False
    text = text.lower()
    return any(marker in text for marker in CONTEXT_MARKERS)

df['has_contextual_grounding'] = df['text'].apply(has_contextual_grounding)

In [26]:
feature_cols = df.select_dtypes(exclude="object").columns.tolist()
feature_cols.pop(0)
label_col = 'effort'

In [27]:
result_feature_v1 = evaluate_feature_impact(df=df, label_col=label_col, feature_cols=feature_cols)

# Feature Analysis – Contextual Grounding

- This feature successfully captured **explicit first-person situational grounding** expressed via lexical markers, such as:
  - *“I am currently…”*
  - *“I work as…”*

- This improved precision for posts where effort is signaled through **stated constraints**.

## Error Analysis

- Systematic blind spots were identified, including:
  - **Interpersonal context**
    - *“my mom is hospitalized”*
  - **Third-party institutional context**
    - *“the dealer told me”*
  - **Temporal narrative context**
    - *“first…, then…”*

- These forms of context require:
  - Relational modeling
  - Temporal structure

- Such signals are **not captured by surface-level lexical markers**.

## Next Iteration Focus

- The next feature iteration will prioritize:
  - **Temporal progression modeling**
- This approach avoids expanding keyword lists and instead:
  - Ensures **orthogonal signal coverage**


### What is **temporal effort**?
Temporal effort is indicated by explicit progression or contrast between past and present states, actions, or belief.

In [28]:
TEMPORAL_MARKERS = [
    # PROGRESSION
    "first", "then", "later", "eventually","initially", "at first",

    # contrast
    "used to", "but now", "now i", "before", "after",

    # persistence
    "have been", "i've been", "so far", "for a while"
]

def has_temporal_progression(text):
    if not isinstance(text, str):
        return False
    text = text.lower()
    return any(marker in text for marker in TEMPORAL_MARKERS)

df['has_temporal_progression'] = df['text'].apply(has_temporal_progression)

In [29]:
feature_cols = df.select_dtypes(exclude="object").columns.tolist()
feature_cols.pop(0)
label_col = 'effort'

In [30]:
result_feature_temporal = evaluate_feature_impact(df=df, label_col=label_col, feature_cols=feature_cols)

In [31]:
result_feature_temporal

,title,body,true_label,pred_without,pred_with,fold,targetted_label
22,I KISSED A WOMAN FOR THE FIRST TIME EVER AT 25!!!,November 29th 2025.\nI actually did it! Someth...,0,0,1,1,has_temporal_progression
37,Denied Entry into Japan.,I know Japan has some of the strictest laws in...,0,0,1,2,has_temporal_progression
116,"Brother-in-law died, has little money. Am I re...",My wife's brother died and his estate consists...,1,0,1,2,has_temporal_progression
174,DAE pick up things with their feet?,"I am not a monkey, I just use my feet to pick ...",0,0,1,2,has_temporal_progression
165,Is the phrase “that’s too bad” meant sincerely?,Brit here. I sometimes see/hear Americans use ...,0,1,0,4,has_temporal_progression


In [32]:
df.loc[165]

title                         Is the phrase “that’s too bad” meant sincerely?
body                        Brit here. I sometimes see/hear Americans use ...
effort                                                                      0
subreddit                                                     r/AskAnAmerican
num_paragraphs                                                              2
has_multi_paragraphs                                                     True
text                        Is the phrase “that’s too bad” meant sincerely...
avg_sentence_length                                                 10.909091
num_tokens                                                                116
has_first_person                                                         True
has_attempted_marker                                                    False
has_constraint_marker                                                   False
question_count                                                  

# Observation for `has_temporal_progression`:
Manual inspection of prediction changes revealed that the temporal grounding feature captures explicit situational framing (e.g., sustained personal actions over time), but occasionally misfires on rhetorical or abstract first-person statements. Additionally, the model exhibited a tendency to associate narrative-style posts without explicit questions with higher effort. Rather than expanding the feature toward semantic interpretation, I intentionally froze it to preserve interpretability and avoid encoding narrative verbosity as a proxy for effort.

# Effort Model — Final Summary

## Goal
To predict whether a post demonstrates **high author effort**, defined as the presence of contextual grounding, constraints, or reflective framing beyond a simple or curiosity-driven question.

## Approach
I implemented a **numerical-only baseline** using interpretable structural and intent-level features, including:

- Post length and paragraph structure  
- Sentence complexity  
- First-person grounding  
- Attempt and constraint markers  
- Contextual and temporal grounding cues  

A **logistic regression model** with stratified cross-validation was used to preserve interpretability and support error-driven iteration.

## Key Findings
- Structural signals (length, paragraphs) capture coarse effort but introduce surface-structure bias.
- Intent features improve precision for *worked-on* posts but misfire on abstract or rhetorical first-person statements.
- Temporal grounding provides limited but controlled improvements in borderline cases.
- Error analysis revealed consistent ambiguity between narrative style and genuine effort intent.

## Limitations
- The model cannot reliably distinguish semantic nuance such as interpersonal storytelling, abstract reflection, or third-party scenarios.
- Remaining errors are largely annotation-bound or require semantic understanding beyond lexical features.

## Conclusion
The effort model successfully captures surface- and intent-level effort in an interpretable manner. Further improvements would require semantic representations (e.g., embeddings), which were intentionally excluded to preserve clarity and avoid over-engineering at this stage.


In [33]:
def features_effort_v1(df):
    import re
    
    df = df.copy()
    
    # -----------------------------
    # Drop columns only if present
    # -----------------------------
    cols_to_drop = ['tag', 'openness', 'id', 'is_confident']
    existing_cols = [c for c in cols_to_drop if c in df.columns]
    if existing_cols:
        df.drop(columns=existing_cols, inplace=True)
    
    # -----------------------------
    # Paragraph features
    # -----------------------------
    def count_paragraphs(text):
        if not isinstance(text, str) or text.strip() == "":
            return 0
        return len([p for p in text.split('\n') if p.strip()])
    
    if 'body' in df.columns:
        df['num_paragraphs'] = df['body'].apply(count_paragraphs)
        df['has_multi_paragraphs'] = df['num_paragraphs'] >= 2
    else:
        df['num_paragraphs'] = 0
        df['has_multi_paragraphs'] = False
    
    # -----------------------------
    # Combine title + body
    # -----------------------------
    if 'title' in df.columns:
        title = df['title'].fillna('')
    else:
        title = ''
        
    if 'body' in df.columns:
        body = df['body'].fillna('')
    else:
        body = ''
    
    df['text'] = title + ' ' + body
    
    # -----------------------------
    # Average sentence length
    # -----------------------------
    def avg_sentence_length(text):
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        if not sentences:
            return 0
        word_counts = [len(s.split()) for s in sentences]
        return sum(word_counts) / len(sentences)
    
    df['avg_sentence_length'] = df['text'].apply(avg_sentence_length)
    
    # -----------------------------
    # Token count
    # -----------------------------
    def count_tokens(text):
        if not isinstance(text, str):
            return 0
        return len(text.split())
    
    df['num_tokens'] = df['text'].apply(count_tokens)
    
    # -----------------------------
    # First person marker
    # -----------------------------
    FIRST_PERSON_PATTERN = re.compile(
        r"\b(i|my|me|mine|i've|i'm|i am)\b",
        re.IGNORECASE
    )
    
    df['has_first_person'] = df['text'].apply(
        lambda x: bool(FIRST_PERSON_PATTERN.search(x)) if isinstance(x, str) else False
    )
    
    # -----------------------------
    # Attempt markers
    # -----------------------------
    ATTEMPT_MARKERS = [
        "tried", "attempted", "failed", 
        "didn't work", "doesn't work", 
        "already tried", "so far"
    ]
    
    def has_attempt_marker(text):
        if not isinstance(text, str):
            return False
        text = text.lower()
        return any(marker in text for marker in ATTEMPT_MARKERS)
    
    df['has_attempted_marker'] = df['text'].apply(has_attempt_marker)
    
    # -----------------------------
    # Constraint markers
    # -----------------------------
    CONSTRAINT_MARKERS = [
        "but", "however", "because", "although", 
        "though", "except", "unless", "while"
    ]
    
    def has_constraint_marker(text):
        if not isinstance(text, str):
            return False
        text = text.lower()
        return any(f" {marker} " in text for marker in CONSTRAINT_MARKERS)
    
    df['has_constraint_marker'] = df['text'].apply(has_constraint_marker)
    
    # -----------------------------
    # Question count
    # -----------------------------
    df['question_count'] = df['text'].str.count(r'\?')
    
    # -----------------------------
    # Context grounding
    # -----------------------------
    CONTEXT_MARKERS = [
        "because", "due to", "as a result", "currently", 
        "right now", "i am", "i'm", "i work", "i live", 
        "based in", "looking for", "trying to", 
        "in my role", "in my situation"
    ]
    
    def has_contextual_grounding(text):
        if not isinstance(text, str):
            return False
        text = text.lower()
        return any(marker in text for marker in CONTEXT_MARKERS)
    
    df['has_contextual_grounding'] = df['text'].apply(has_contextual_grounding)
    
    # -----------------------------
    # Temporal progression
    # -----------------------------
    TEMPORAL_MARKERS = [
        "first", "then", "later", "eventually", "initially", "at first",
        "used to", "but now", "now i", "before", "after",
        "have been", "i've been", "so far", "for a while"
    ]
    
    def has_temporal_progression(text):
        if not isinstance(text, str):
            return False
        text = text.lower()
        return any(marker in text for marker in TEMPORAL_MARKERS)
    
    df['has_temporal_progression'] = df['text'].apply(has_temporal_progression)
    
    return df


In [34]:
df.columns

Index(['title', 'body', 'effort', 'subreddit', 'num_paragraphs',
       'has_multi_paragraphs', 'text', 'avg_sentence_length', 'num_tokens',
       'has_first_person', 'has_attempted_marker', 'has_constraint_marker',
       'question_count', 'has_contextual_grounding',
       'has_temporal_progression'],
      dtype='object')